# ISLP Chapter 6 Lab
#### Author: Thomas Fitzgerald
This notebook contains my work for the lab accompanying Chapter 6 of [*An Introduction to Statistical Learning with Python* ("*ISLP*")](https://www.statlearning.com/). The sixth chapter of *ISLP* covers linear model selection and regularization (ex. subset selection, training error adjustment, shrinkage methods, and principal companent analysis). This lab makes use of libraries introduced in the previous lab (**numpy**, **statsmodel**, **ISLP**, **sklearn**, and **functools**) as well as portions of the **10bnb** library. To begin, the imports from the previous labs are pulled in:

In [1]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
from statsmodels.api import OLS
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from functools import partial

Next, imports new to this lab are pulled in:

In [2]:
from sklearn.cross_decomposition import PLSRegression
from ISLP.models import (
    Stepwise,
    sklearn_selected,
    sklearn_selection_path
)
from l0bnb import fit_path

### Subset Selection Methods
This section implements methods for reducing the number of variables included in a model by restricting the model to a subset of the initial variables.

##### Forward Selection
To begin, the forward-selection approach is applied to the **Hitters** dataset. The goal is predict a player's **Salary** based on various statistics associated with their prior-year performance. It is important to note that the data must be cleaned as the **Salary** field is missing for some of the players. The **np.isnan()** function can be used to identofy missing observations and then the **arr.dropna()** method can be used to remove those observations.

In [3]:
Hitters = load_data('Hitters')
print("Missing Salaries:", np.isnan(Hitters['Salary']).sum())
Hitters = Hitters.dropna()
Hitters.shape

Missing Salaries: 59


(263, 20)

With the data cleaned, an optimal model based on forward selection and $C_p$ can be determined. The $C_p$ score is not a built-in component of the **sklearn** library, so it must be defined herein. Additionally, **sklearn** seeks to maximize a score, so the function is constructed to determine the negative $C_p$ statistic.

In [4]:
def nCp(sigma2, estimator, X, Y):
    "Return negative Cp statistic"
    n, p = X.shape
    Y_hat = estimator.predict(X)
    RSS = np.sum((Y - Y_hat)**2)
    return -(RSS + 2 * p * sigma2) / n

To use the scoring function above, it is required that the residual variance ($\sigma^2$) is estimated. This estimate is obtained by fitting the biggest model, including all variables, and etimating $\sigma^2$ based on its MSE.

In [7]:
design = MS(Hitters.columns.drop('Salary')).fit(Hitters)
Y = np.array(Hitters['Salary'])
X = design.transform(Hitters)
sigma2 = OLS(Y, X).fit().scale

As the **sklearn_selected()** function expects a scorer with only the last three arguments of **nCp()** the **partial()** function is used to freeze the **sigma2** argument to the above estimate of $\sigma^2$.

In [8]:
neg_Cp = partial(nCp, sigma2)

Along with this properly formatted scorer, a search strategy needs to be specified. For this, the **Stepwise()** function from the **ISLP.models** package is used. The **Stepwise.first_peak()** method runs forward stepwise until any further additions to the model do not result in an improvement in the evaluation score. The **stepwise.fixed_steps()** method runs a fixed number of steps of stepwise search.

In [10]:
strategy = Stepwise.first_peak(
    design,
    direction='forward',
    max_terms=len(design.terms)
)

Now, the linear regression model with **Salary** as the outcome can be fit using forward stepwise selection. To do so, the **sklear_selection()** function from the **ISLP.models** package is used. This takes a model from **statsmodels** along with a search strategy and selects a model with its **fit** method. With no **scoring** argument passed, the score defaults to MSE and all variables will be selected.

In [12]:
hitters_MSE = sklearn_selected(OLS, strategy)
hitters_MSE.fit(Hitters, Y)
len(hitters_MSE.selected_state_)

19

In [ ]:
all_predictors = Hitters.columns.drop('Salary')
len(all_predictors)

19

The two cells above demonstrate the selection using MSE and the fact that all predictors are included when the scoring method is MSE. 

##### Choosing Among Models Using the Validation Set Approach and Cross-Validation
Cross-validation presents an alternative to using $C_p$ for selecting a model with forward selection. Cross-validation requires a method that stores the full path of models found in forward selection and allows prediction for each. The **sklearn_selection_path()** function from the **ISLP.models** package provides such functionality. **ISLP.models** aditionally provides the **cross_val_predict()** function to compute the cross-validation predictions for each of the models. These predcitions can then be used to compute the cross-validated MSE along the forward selection path.

Below, a strategy is defined that fits the full forward selection path. The **sklearn_selection_path()** takes several arguments. In this case, the defaults are used to to select the model at each step that provides the greatest reduction in RSS.

In [ ]:
strategy = Stepwise.fixed_steps(
    design,
    len(design.terms),
    directions='forward'
)
full_path = sklearn_selection_path(OLS, strategy)

With the strategy defined, the full path can be fit on the **Hitters** data as follows.

In [ ]:
full_path.fit(Hitters, Y)
Yhat_in = full_path.predict(Hitters)
Yhat_in.shape